In [6]:
# %matplotlib inline
%config InlineBackend.figure_format = "retina"
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
})

import seaborn as sns
sns.set_theme(context="talk", style="whitegrid", 
              palette="colorblind", color_codes=True, 
              rc={"figure.figsize": [12, 8]})

import yfinance as yf
import numpy as np
import pandas as pd
import quantstats as qs

import requests
import csv

from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

import statsmodels.api as sm

from pypfopt import black_litterman
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier

In [7]:
# Carteira teorica do Idiv

url = "https://raw.githubusercontent.com/BDonadelli/Finance-playground/main/data/Cart_Idiv.csv"

response = requests.get(url)
response.encoding = 'utf-8'  # ou 'latin-1' se necessário

# Divide o conteúdo em linhas
linhas = response.text.splitlines()

# Ignora as duas primeiras e duas últimas linhas
linhas_filtradas = linhas[2:-2]

# Extrai a primeira coluna de cada linha restante
ASSETS = []
for linha in linhas_filtradas:
    # O separador é ";", pega o primeiro campo
    campos = linha.split(';')
    if campos:  # garante que a linha não está vazia
        ASSETS.append((campos[0],campos[4]))

ASSETS = [(ticker, float(str(value).replace(',', '.'))) for ticker, value in ASSETS]

try:
    pesos_indice = [value for ticker, value in ASSETS]
    tickers = [ticker+'.SA' for ticker, value in ASSETS]
    tickers.sort()
except :    
    tickers = [ticker+'.SA' for ticker in ASSETS]

tickers.append('AMBP3.SA')
tickers.sort()

print(tickers)

['ABCB4.SA', 'AGRO3.SA', 'ALOS3.SA', 'AMBP3.SA', 'BBAS3.SA', 'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BRAP4.SA', 'BRBI11.SA', 'BRSR6.SA', 'CMIG4.SA', 'CMIN3.SA', 'CPFE3.SA', 'CSMG3.SA', 'CURY3.SA', 'CXSE3.SA', 'DIRR3.SA', 'EGIE3.SA', 'EVEN3.SA', 'EZTC3.SA', 'FESA4.SA', 'FLRY3.SA', 'GRND3.SA', 'ISAE4.SA', 'ITSA4.SA', 'ITUB3.SA', 'ITUB4.SA', 'JHSF3.SA', 'KEPL3.SA', 'KLBN11.SA', 'LAVV3.SA', 'LEVE3.SA', 'LOGG3.SA', 'MBRF3.SA', 'ODPV3.SA', 'PETR3.SA', 'PETR4.SA', 'PGMN3.SA', 'POMO4.SA', 'RANI3.SA', 'RECV3.SA', 'SAPR11.SA', 'SLCE3.SA', 'SYNE3.SA', 'TAEE11.SA', 'TGMA3.SA', 'TIMS3.SA', 'UNIP6.SA', 'VALE3.SA', 'VBBR3.SA', 'VLID3.SA', 'VULC3.SA']


In [8]:
ASSETS = [
    "ABEV3.SA",
    "VLID3.SA",
    "OFSA3.SA",
    "TECN3.SA",
    "SOND5.SA",
    "MDNE3.SA",
    "CURY3.SA",
    "RANI3.SA",
    "JHSF3.SA",
    "LPSB3.SA",
    "MULT3.SA",
    "ITSA4.SA",
    "RECV3.SA",
    "CSUD3.SA",
    "LUXM4.SA",
    "FIQE3.SA",
    "BLAU3.SA",
    "SHUL4.SA",
    "EALT4.SA",
    "RSUL4.SA",
    "POMO4.SA",
    "PETR4.SA",
    "SBSP3.SA",
    "CSMG3.SA",
    "WIZC3.SA",
    "MILS3.SA",
    "CAMB3.SA",
    "VULC3.SA",
    "GRND3.SA"
]

#### Parâmetros

In [9]:
rf = 0.14               # taxa livre de risco
n_days=252              # dias no ano do calendario financeiro, assumindo dados diários pegos no Yahoo Finance
#n_monte_carlo = 10**6 # quantidade de carteiras na simulação

# Definição do período e download dos dados
data_inicio = '2018-01-01'
data_fim = '2026-04-30'

#### preços de fechamento

Baixa dados e limpa a base

In [10]:
prices = yf.download(tickers, start=data_inicio, end=data_fim , auto_adjust=True)['Close']
prices.columns = [col.replace('.SA', '') for col in prices.columns]

benchm = yf.download('^BVSP', start=data_inicio, end=data_fim , auto_adjust=True)['Close']

[*********************100%***********************]  53 of 53 completed
[*********************100%***********************]  1 of 1 completed


In [11]:
# Empresas com mais de 'limiar' dados faltantes
limiar = 10
missing = prices.isna().sum()
empresas_missing = missing[missing > limiar].index.tolist()

print(missing[missing > limiar].sort_values(ascending=False))

tickers = [item for item in ASSETS if item not in empresas_missing]



BRBI11    872
RECV3     824
CXSE3     821
CMIN3     774
CURY3     674
LAVV3     663
PGMN3     663
AMBP3     625
LOGG3     242
VBBR3      49
dtype: int64


In [12]:
import plotly.express as px

if empresas_missing:
    fig = px.line(
        prices[empresas_missing].reset_index(),
        x='Date',
        y=empresas_missing,
        title='Empresas com mais de 10 dados faltantes',
        labels={'value': 'Preço', 'Date': 'Data', 'variable': 'Empresa'}
    )

    fig.show()

In [13]:
# mantem colunas (axis=1) ue possuem no mínimo len(prices) - 20 valores não-nulos.
prices = prices.dropna(axis=1, thresh=len(prices) - 10)
# preenche dados faltantes repetindo ultimo valor
prices = prices.ffill()

prices

,ABCB4,AGRO3,ALOS3,BBAS3,BBDC3,BBDC4,BBSE3,BRAP4,BRSR6,CMIG4,...,SAPR11,SLCE3,SYNE3,TAEE11,TGMA3,TIMS3,UNIP6,VALE3,VLID3,VULC3
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,9.208960,6.842369,18.630674,9.456817,11.138400,12.179819,13.367786,4.202160,7.811344,1.498756,...,12.398582,3.451245,1.566560,9.288571,11.868077,7.978173,5.058343,21.669312,11.916832,5.375476
2018-01-03,9.247839,6.836968,18.772039,9.577435,11.185784,12.235805,13.377099,4.241883,7.848087,1.485834,...,12.291392,3.481202,1.650874,9.301555,11.868077,7.984216,5.428317,21.539463,12.096825,5.641647
2018-01-04,9.220070,7.058387,18.727396,9.669333,11.389474,12.436576,13.405048,4.362473,7.895336,1.468608,...,12.192608,3.467472,1.720012,9.154389,12.324998,7.947949,5.370699,21.627760,12.096825,5.786304
2018-01-05,9.353372,7.204198,18.749718,9.669333,11.392926,12.507017,13.493546,4.467455,7.999000,1.470761,...,12.285085,3.532378,1.720012,9.197675,12.461477,8.014437,5.455612,21.965372,12.072000,5.786304
2018-01-08,9.442239,7.236602,18.228891,9.692307,11.392926,12.503499,13.572728,4.545483,8.132940,1.475068,...,12.211524,3.513655,1.736875,9.154389,12.461477,7.905642,5.701252,22.453604,12.040966,5.815234
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,25.160000,19.680000,31.558214,23.000000,17.191864,19.949961,34.270000,24.059999,16.155130,13.088566,...,42.849998,17.530001,4.060000,43.116379,32.939999,26.049999,60.869999,85.970001,19.563784,16.120001
2026-04-24,25.280001,19.850000,31.281128,22.700001,17.091970,19.900011,34.290001,23.950001,15.876423,12.804031,...,43.610001,17.370001,4.000000,42.585388,32.230000,26.100000,60.540001,85.870003,19.652891,15.940000
2026-04-27,24.930000,19.139999,30.944666,22.510000,16.952118,19.710201,33.950001,23.850000,15.687300,12.558743,...,41.549999,17.240000,3.940000,42.102669,32.250000,25.770000,59.880001,85.500000,19.246962,15.780000


In [14]:
returns = prices.pct_change().dropna()

#log retorno para matriz de covriancia ledoit wolf, tomar cuidado com parametros das bibliotecas

print(len(prices),len(returns),len(tickers),len(ASSETS),len(prices.columns))

2068 2067 29 29 43


Matriz covariância por Ledoit Wolf

In [15]:
from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

"""
cov_matrix é uma NxN matriz de covariância
tickers é um dicionário com os ativos
"""

pesos = [1 / len(ASSETS)] * len(ASSETS)
# dicionario_pesos = dict(zip(tickers, pesos))

#cov = np.cov(market_prices)

model = LedoitWolf()
cov_matrix = model.fit(returns).covariance_
df_cov_matrix = pd.DataFrame(cov_matrix, index=returns.columns, columns=returns.columns)

print(df_cov_matrix.shape)
#print(dicionario_pesos)
print(tickers)

tickers_validos = list(df_cov_matrix.columns)

pesos = np.repeat(1/len(tickers_validos), len(tickers_validos))

dicionario_pesos = dict(zip(tickers_validos, pesos))



(43, 43)
['ABEV3.SA', 'VLID3.SA', 'OFSA3.SA', 'TECN3.SA', 'SOND5.SA', 'MDNE3.SA', 'CURY3.SA', 'RANI3.SA', 'JHSF3.SA', 'LPSB3.SA', 'MULT3.SA', 'ITSA4.SA', 'RECV3.SA', 'CSUD3.SA', 'LUXM4.SA', 'FIQE3.SA', 'BLAU3.SA', 'SHUL4.SA', 'EALT4.SA', 'RSUL4.SA', 'POMO4.SA', 'PETR4.SA', 'SBSP3.SA', 'CSMG3.SA', 'WIZC3.SA', 'MILS3.SA', 'CAMB3.SA', 'VULC3.SA', 'GRND3.SA']


In [16]:
delta = black_litterman.market_implied_risk_aversion(prices)
prior = black_litterman.market_implied_prior_returns(
    dicionario_pesos,
    delta,
    cov_matrix
)
print(delta)

ABCB4     1.777735
AGRO3     2.011665
ALOS3     0.912510
BBAS3     1.300449
BBDC3     0.932728
BBDC4     0.966698
BBSE3     2.321505
BRAP4     2.204131
BRSR6     1.226214
CMIG4     2.666147
CPFE3     2.861778
CSMG3     2.605377
DIRR3     2.237331
EGIE3     2.913165
EVEN3     0.752133
EZTC3     0.630331
FESA4     1.167987
FLRY3     0.321384
GRND3     0.549470
ISAE4     3.761439
ITSA4     2.427835
ITUB3     2.534815
ITUB4     1.915717
JHSF3     1.851542
KEPL3     1.938175
KLBN11    1.258539
LEVE3     1.699174
MBRF3     1.171516
ODPV3     0.967243
PETR3     2.150694
PETR4     2.226632
POMO4     1.389336
RANI3     1.247864
SAPR11    2.085161
SLCE3     2.054291
SYNE3     0.870921
TAEE11    5.249976
TGMA3     1.234114
TIMS3     2.061365
UNIP6     2.270381
VALE3     1.759532
VLID3     0.749243
VULC3     1.335835
dtype: float64


/home/caio/Documentos/ic_26/.venv/lib/python3.12/site-packages/pypfopt/black_litterman.py:45: RuntimeWarning: If cov_matrix is not a dataframe, market cap index must be aligned to cov_matrix
  warnings.warn(


In [17]:
print(prior)

ABCB4     0.000309
AGRO3     0.000204
ALOS3     0.000206
BBAS3     0.000291
BBDC3     0.000196
BBDC4     0.000204
BBSE3     0.000271
BRAP4     0.000311
BRSR6     0.000236
CMIG4     0.000538
CPFE3     0.000387
CSMG3     0.000449
DIRR3     0.000523
EGIE3     0.000340
EVEN3     0.000217
EZTC3     0.000179
FESA4     0.000180
FLRY3     0.000053
GRND3     0.000089
ISAE4     0.000392
ITSA4     0.000432
ITUB3     0.000418
ITUB4     0.000347
JHSF3     0.000482
KEPL3     0.000249
KLBN11    0.000106
LEVE3     0.000268
MBRF3     0.000205
ODPV3     0.000109
PETR3     0.000467
PETR4     0.000480
POMO4     0.000288
RANI3     0.000196
SAPR11    0.000317
SLCE3     0.000213
SYNE3     0.000165
TAEE11    0.000496
TGMA3     0.000282
TIMS3     0.000287
UNIP6     0.000421
VALE3     0.000246
VLID3     0.000163
VULC3     0.000280
dtype: float64


In [18]:
fatores_nefin = pd.read_csv('nefin_factors.csv')

# A coluna Date já existe, só converter e setar como índice
fatores_nefin['Date'] = pd.to_datetime(fatores_nefin['Date'])
fatores_nefin.set_index('Date', inplace=True)

# Reamostrar os fatores diários para mensais
#fatores_mensais = fatores_nefin.resample('ME').sum()
#print(fatores_mensais)

fatores_mensais = (1 + fatores_nefin).resample('ME').prod() - 1
print(fatores_mensais)
#verificar se a taxa e fatores estao sendo colocadas certas mensalmente

# Alinhar com os retornos
datas_comuns = returns.index.intersection(fatores_mensais.index)
retornos_alinhados = returns.loc[datas_comuns]
fatores_alinhados = fatores_mensais.loc[datas_comuns]

                     Unnamed: 0  Rm_minus_Rf       SMB       HML       WML  \
Date                                                                         
2001-01-31 -1250660718674968577     0.139540  0.163653  0.147510 -0.012725   
2001-02-28   231432332370247679    -0.085317  0.052359  0.022571  0.071785   
2001-03-31 -2890597822505680897    -0.077331 -0.014624  0.060201  0.076311   
2001-04-30  7254507651604676607     0.026857 -0.120024 -0.155679 -0.049503   
2001-05-31  8983196893043490815    -0.003461 -0.094637 -0.154272 -0.017923   
...                         ...          ...       ...       ...       ...   
2025-12-31 -8644949316997480449     0.004598 -0.031421  0.009458 -0.028194   
2026-01-31  4353516797392060415     0.103267 -0.014013  0.043407  0.023017   
2026-02-28  3190475155598606335     0.026488 -0.044621 -0.033540  0.007398   
2026-03-31 -8315929580154126337    -0.021942 -0.039934  0.004906 -0.021615   
2026-04-30             39181339     0.001203  0.005243  0.009763

In [19]:
print(pesos)

[0.02325581 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581
 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581
 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581
 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581
 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581
 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581
 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581 0.02325581
 0.02325581]


In [32]:
# -------------------------------------------------------
# CORREÇÃO 1: Remover coluna lixo do CSV
# -------------------------------------------------------
fatores_nefin = pd.read_csv('nefin_factors.csv')

# Remover qualquer coluna 'Unnamed'
fatores_nefin = fatores_nefin.loc[:, ~fatores_nefin.columns.str.contains('^Unnamed')]

# Resto do processamento (igual ao anterior)
fatores_nefin['Date'] = pd.to_datetime(fatores_nefin['Date'])
fatores_nefin.set_index('Date', inplace=True)
fatores_mensais = (1 + fatores_nefin).resample('ME').prod() - 1

# Verificar colunas
print("Colunas dos fatores:", fatores_mensais.columns.tolist())
# Esperado: ['Rm_minus_Rf', 'SMB', 'HML', 'WML', 'IML', 'Risk_Free']

# -------------------------------------------------------
# CORREÇÃO 2 e 3: Alinhar escalas (tudo mensal)
# -------------------------------------------------------

# Se 'returns' está em frequência diária, converta para mensal
# (assumindo que returns vem de prices diários)
returns_mensal = (1 + returns).resample('ME').prod() - 1

# Realinhar com fatores
datas_comuns       = returns_mensal.index.intersection(fatores_mensais.index)
retornos_alinhados = returns_mensal.loc[datas_comuns]
fatores_alinhados  = fatores_mensais.loc[datas_comuns]

# Recalcular cov_matrix em frequência MENSAL
from sklearn.covariance import LedoitWolf
model        = LedoitWolf()
cov_mensal   = model.fit(retornos_alinhados).covariance_
df_cov_matrix = pd.DataFrame(cov_mensal, index=retornos_alinhados.columns,
                             columns=retornos_alinhados.columns)

# -------------------------------------------------------
# Recalcular prior em escala mensal
# -------------------------------------------------------
# Opção A: usar prices mensais
prices_mensal = prices.resample('ME').last()
delta = black_litterman.market_implied_risk_aversion(prices_mensal, frequency=12)

# Opção B (mais simples e correta para BL no Brasil):
# usar delta fixo ~3.07 como recomendado pelo Idzorek (2007) e o próprio artigo
delta = 3.07

prior = black_litterman.market_implied_prior_returns(
    dicionario_pesos, delta, df_cov_matrix
)

print("\nPrior em escala mensal:")
print(prior.describe())

# -------------------------------------------------------
# Refazer o resto do código
# (regressão OLS, retornos estimados, BL)
# -------------------------------------------------------
col_rf       = 'Risk_Free'
fatores_cols = [c for c in fatores_alinhados.columns if c != col_rf]

# Verificação final antes da regressão
print("\nFatores que vão para o OLS:", fatores_cols)
print("Médias dos fatores (sanity check):")
print(fatores_alinhados[fatores_cols].mean())

Colunas dos fatores: ['Rm_minus_Rf', 'SMB', 'HML', 'WML', 'IML', 'Risk_Free']

Prior em escala mensal:
count    43.000000
mean      0.009907
std       0.003744
min       0.001859
25%       0.006710
50%       0.010059
75%       0.012077
max       0.019546
dtype: float64

Fatores que vão para o OLS: ['Rm_minus_Rf', 'SMB', 'HML', 'WML', 'IML']
Médias dos fatores (sanity check):
Rm_minus_Rf    0.003451
SMB            0.002155
HML            0.011700
WML            0.012779
IML            0.001813
dtype: float64


In [33]:
lista_tickers = tickers_validos

# -------------------------------------------------------
# ETAPA 1: OLS — regressão Fama-French por ativo
col_rf       = 'Risk_Free'
fatores_cols = [c for c in fatores_alinhados.columns if c != col_rf]
X_ols         = sm.add_constant(fatores_alinhados[fatores_cols])
medias_fatores = fatores_alinhados[fatores_cols].mean()

betas_dict = {}
for ticker in lista_tickers:
    Y      = retornos_alinhados[ticker] - fatores_alinhados[col_rf]
    modelo = sm.OLS(Y, X_ols).fit()
    betas_dict[ticker] = modelo.params

betas_df = pd.DataFrame(betas_dict).T

# ETAPA 2: Retornos estimados r̂_i (Eq. 12)
retornos_estimados = (
    betas_df['const']
    + betas_df[fatores_cols].dot(medias_fatores)
)

In [34]:
#Abordagem intuitiva, fazer opinião absoluta (^r) para cada ativo e usar como vetor q, ver no claude a ref academica 

# ETAPA 1 e 2 iguais ao código atual
# (regressão OLS + retornos estimados r̂_i)

# ETAPA 3 (nova): visões absolutas por ativo
# Cada ativo tem sua própria visão = r̂_i previsto pelo FF

# P_t = matriz identidade (cada visão refere-se a um único ativo)
P_matrix = np.eye(len(lista_tickers))

# Q_vector = retornos estimados pelo FF
Q_vector = retornos_estimados[lista_tickers].values

# Ω = diagonal com a variância de cada previsão
# Opção 1: variância dos resíduos da regressão (mais teórica)
variancias_residuos = {}
for ticker in lista_tickers:
    Y = retornos_alinhados[ticker] - fatores_alinhados[col_rf]
    modelo = sm.OLS(Y, X_ols).fit()
    variancias_residuos[ticker] = modelo.mse_resid

Omega_matrix = np.diag([variancias_residuos[t] for t in lista_tickers])

# Opção 2 (mais simples, à la He-Litterman): Ω = τ * diag(P Σ P^T)
# Sigma = df_cov_matrix.loc[lista_tickers, lista_tickers].values
# Omega_matrix = tau * np.diag(np.diag(P_matrix @ Sigma @ P_matrix.T))

# ETAPA 4: BL
bl = BlackLittermanModel(
    df_cov_matrix.loc[lista_tickers, lista_tickers],
    pi=prior[lista_tickers],
    Q=Q_vector,
    P=P_matrix,
    omega=Omega_matrix,
    tau= 0.1
)

#cov_matrix_bm = df_cov_matrix.loc[tickers, tickers]


mu_BL = bl.bl_returns()
print("\nRetornos posteriores BL:")
print(mu_BL.sort_values(ascending=False))

ef = EfficientFrontier(mu_BL, cov_matrix, weight_bounds=(0, 0.10))
ef.max_sharpe()
weights = ef.clean_weights()
print("\nPesos ótimos (max Sharpe):")
print(pd.Series(weights).sort_values(ascending=False))


Retornos posteriores BL:
EVEN3     0.016124
JHSF3     0.015935
PETR3     0.012933
EZTC3     0.012761
PETR4     0.012695
TGMA3     0.012270
DIRR3     0.012138
RANI3     0.011651
VLID3     0.011484
CMIG4     0.010615
POMO4     0.010374
KEPL3     0.010352
VULC3     0.010208
CSMG3     0.010013
BBAS3     0.009922
UNIP6     0.009736
ALOS3     0.009597
BRSR6     0.009464
BBDC4     0.009339
SYNE3     0.009248
BBDC3     0.009038
MBRF3     0.008707
ITUB4     0.008227
ITSA4     0.008091
ABCB4     0.008055
SAPR11    0.007764
ITUB3     0.007648
BRAP4     0.007601
AGRO3     0.007577
LEVE3     0.007245
FESA4     0.006863
CPFE3     0.006467
GRND3     0.006079
TIMS3     0.006018
TAEE11    0.006007
ISAE4     0.005843
EGIE3     0.005343
BBSE3     0.005184
VALE3     0.005147
FLRY3     0.004265
ODPV3     0.004026
KLBN11    0.003978
SLCE3     0.003742
dtype: float64

Pesos ótimos (max Sharpe):
AGRO3     0.10000
KEPL3     0.10000
JHSF3     0.10000
TAEE11    0.10000
ISAE4     0.10000
CSMG3     0.08419
PETR3 

In [38]:
from pypfopt import EfficientFrontier, risk_models, exceptions
rf_media = fatores_alinhados[col_rf].mean()

print(f"rf_media (mensal): {rf_media:.6f}")
print(f"rf_media (anual):  {(1+rf_media)**12 - 1:.4f}\n")

print("Médias dos fatores (mensais):")
print(medias_fatores)
print()

print("Retornos estimados FF (antes do BL):")
print(retornos_estimados.describe())
print(f"\nQuantos retornos FF > rf: {(retornos_estimados > rf_media).sum()}")
print()

print("Prior CAPM:")
print(prior[lista_tickers].describe())
print(f"\nQuantos priors > rf: {(prior[lista_tickers] > rf_media).sum()}")
print()

print("Retornos posteriores BL:")
print(mu_BL.describe())
print(f"\nAcima de rf ({rf_media:.6f}): {(mu_BL > rf_media).sum()} de {len(mu_BL)}")

# 1. Diagnóstico
print("=== DIAGNÓSTICO ===")
print(f"Retornos posteriores BL:\n{mu_BL.describe()}\n")
print(f"Acima de rf ({rf_media:.6f}): {(mu_BL > rf_media).sum()} de {len(mu_BL)}")
print(f"Condition number cov: {np.linalg.cond(cov_matrix):.2e}")

# 2. Garantir matriz PSD
cov_matrix_bm_psd = risk_models.fix_nonpositive_semidefinite(cov_matrix)

# 3. Tentar max_sharpe com rf correto
try:
    ef = EfficientFrontier(mu_BL, cov_matrix_bm_psd)
    ef.max_sharpe(risk_free_rate=rf_media)
    weights = ef.clean_weights()
    print("\n✅ max_sharpe convergiu")
except exceptions.OptimizationError as e:
    print(f"\n⚠️ max_sharpe falhou: {e}")
    print("Tentando min_volatility como fallback...")
    
    ef = EfficientFrontier(mu_BL, cov_matrix_bm_psd)
    ef.min_volatility()
    weights = ef.clean_weights()
    print("✅ min_volatility convergiu")

print("\nPesos finais:")
print(pd.Series(weights).sort_values(ascending=False).head(15))

# Performance esperada
ef.portfolio_performance(risk_free_rate=rf_media, verbose=True)

print("na verdade, o valor acima é MENSAL!")

rf_media (mensal): 0.007025
rf_media (anual):  0.0876

Médias dos fatores (mensais):
Rm_minus_Rf    0.003451
SMB            0.002155
HML            0.011700
WML            0.012779
IML            0.001813
dtype: float64

Retornos estimados FF (antes do BL):
count    43.000000
mean      0.010060
std       0.007954
min      -0.005507
25%       0.005373
50%       0.008332
75%       0.013563
max       0.029276
dtype: float64

Quantos retornos FF > rf: 28

Prior CAPM:
count    43.000000
mean      0.009907
std       0.003744
min       0.001859
25%       0.006710
50%       0.010059
75%       0.012077
max       0.019546
dtype: float64

Quantos priors > rf: 31

Retornos posteriores BL:
count    43.000000
mean      0.008739
std       0.003035
min       0.003742
25%       0.006273
50%       0.008707
75%       0.010363
max       0.016124
dtype: float64

Acima de rf (0.007025): 30 de 43
=== DIAGNÓSTICO ===
Retornos posteriores BL:
count    43.000000
mean      0.008739
std       0.003035
min       0

In [37]:
# 2. Anualizar prior e retornos do FF
prior_anual = (1 + prior) ** 12 - 1
mu_BL_anual = (1 + mu_BL) ** 12 - 1
rf_anual = (1 + rf_media) ** 12 - 1

cov_mensal = LedoitWolf().fit(retornos_alinhados).covariance_
cov_matrix_anual = pd.DataFrame(
    cov_mensal * 12,                    # anualizar
    index=retornos_alinhados.columns,
    columns=retornos_alinhados.columns
)

# 3. Otimização com tudo em escala anual
ef = EfficientFrontier(mu_BL_anual, cov_matrix_anual)
ef.max_sharpe(risk_free_rate=rf_anual)
weights = ef.clean_weights()
ef.portfolio_performance(risk_free_rate=rf_anual, verbose=True)

Expected annual return: 19.0%
Annual volatility: 35.7%
Sharpe Ratio: 0.29


(np.float64(0.1897982577755943),
 np.float64(0.3570616704481673),
 np.float64(0.2861298573860186))

In [39]:
#ABORDAGEM KO et al. do grid 1 uma visão relativa 
#ideia: small/value vai superar big/growth, argumenta no artigo ser mais robusta empiricamente.


# ETAPA 3: Grid nxn
book_to_market = {}
for ticker in lista_tickers:
    info = yf.Ticker(ticker + '.SA').info
    pb   = info.get('priceToBook', None)
    book_to_market[ticker] = 1 / pb if pb else None
book_to_market = pd.Series(book_to_market).dropna()

# ✅ market_cap criado ANTES de filtrar
market_cap = pd.Series(
    {ticker: peso for ticker, peso in zip(lista_tickers, pesos)}
)

# ✅ filtrar AMBOS para tickers com BM disponível
tickers_bm     = [t for t in lista_tickers if t in book_to_market.index]
market_cap     = market_cap[tickers_bm]      # ← corrigido (era market_cap[tickers_bm] sem definir antes)
book_to_market = book_to_market[tickers_bm]

# ✅ size_quintil calculado sobre market_cap já filtrado
n            = 3
size_quintil = pd.qcut(market_cap.rank(method='first'), n, labels=False)
bm_labels    = pd.Series(index=tickers_bm, dtype=int)

for g in range(n):
    grupo_tickers = size_quintil[size_quintil == g].index  # ← agora alinhado com tickers_bm
    bm_labels[grupo_tickers] = pd.qcut(
        book_to_market[grupo_tickers].rank(method='first'),
        n, labels=False, duplicates='drop'
    )

grupo_1  = [t for t in tickers_bm if size_quintil[t] == 0 and bm_labels[t] == n-1]
grupo_9 = [t for t in tickers_bm if size_quintil[t] == n-1 and bm_labels[t] == 0]

if not grupo_1 or not grupo_9:
    raise ValueError("Grupo 1 ou 25 vazio — verifique os dados de size/BM")


print(f"Grupo 1 (small/value): {grupo_1}")
print(f"Grupo 9 (big/growth):  {grupo_9}")

# ETAPA 4: P_t (Eq. 11)
P_t           = pd.Series(0.0, index=tickers_bm)
P_t[grupo_1] =  1 / len(grupo_1)
P_t[grupo_9] = -1 / len(grupo_9)

# ETAPA 5: q_t (Eq. 13-15)
q_t = retornos_estimados[grupo_1].mean() - retornos_estimados[grupo_9].mean()
print(f"\nView return q_t: {q_t:.6f}")

# print("Médias dos fatores (mensais):")
# print(medias_fatores)
# print(f"\nMédia Rf: {rf_media:.6f}")

# 2. Compare q_t com a magnitude típica dos retornos
print(f"\nq_t / média dos retornos estimados: {q_t / retornos_estimados.mean():.2%}")

# 3. Veja a dispersão de retornos_estimados
print(f"\nRetornos estimados FF:")
print(retornos_estimados.describe())
# ETAPA 6: Ω_t (Eq. 16)

# -------------------------------------------------------
# ETAPA 6: Incerteza da visão Ω_t (Eq. 16)  τ = 0.1
# -------------------------------------------------------
# tau     = 0.1
# Sigma   = df_cov_matrix.loc[tickers_bm, tickers_bm].values
# p1      = P_t.values.reshape(1, -1)
# omega_t = float(tau * (p1 @ Sigma @ p1.T))

# # -------------------------------------------------------
# # ETAPA 7: Retornos posteriores Black-Litterman (Eq. 7-8)
# # -------------------------------------------------------
# pi      = prior[tickers_bm].values.reshape(-1, 1)
# P       = p1
# q       = np.array([[q_t]])
# Omega   = np.array([[omega_t]])

# Sigma_BL = np.linalg.inv(
#     np.linalg.inv(tau * Sigma) + P.T @ np.linalg.inv(Omega) @ P
# )

# mu_BL = Sigma_BL @ (
#     np.linalg.inv(tau * Sigma) @ pi
#     + P.T @ np.linalg.inv(Omega) @ q
# )

# mu_BL_series = pd.Series(mu_BL.flatten(), index=tickers_bm)
# print(mu_BL_series.sort_values(ascending=False))

Grupo 1 (small/value): ['ABCB4', 'AGRO3', 'BBAS3', 'BBDC3', 'BRSR6']
Grupo 9 (big/growth):  ['ODPV3', 'POMO4', 'TGMA3', 'TIMS3', 'UNIP6']

View return q_t: -0.006100

q_t / média dos retornos estimados: -60.63%

Retornos estimados FF:
count    43.000000
mean      0.010060
std       0.007954
min      -0.005507
25%       0.005373
50%       0.008332
75%       0.013563
max       0.029276
dtype: float64


In [40]:
tau     = 0.1
Sigma   = df_cov_matrix.loc[tickers_bm, tickers_bm].values
p1      = P_t.values.reshape(1, -1)


# ✅ Correto — .item() extrai o escalar de um array (1,1)
omega_t = (tau * (p1 @ Sigma @ p1.T)).item()

# Verificação de sanidade
print(f"q_t    : {q_t:.6f}")
print(f"omega_t: {omega_t:.6f}")
print(f"P_t não-zero: {P_t[P_t != 0]}")

# Preparar para BlackLittermanModel
P_matrix     = p1                        # (1, N)
Q_vector     = np.array([q_t])           # (1,)
Omega_matrix = np.array([[omega_t]])     # (1, 1)
cov_matrix_bm = df_cov_matrix.loc[tickers_bm, tickers_bm]
prior_bm      = prior[tickers_bm]

# Black-Litterman
bl = BlackLittermanModel(
    cov_matrix_bm,
    pi=prior_bm,
    Q=Q_vector,
    P=P_matrix,
    omega=Omega_matrix,
    tau=tau
)

mu_BL = bl.bl_returns()
print("\nRetornos posteriores BL:")
print(mu_BL.sort_values(ascending=False))

ef = EfficientFrontier(mu_BL, cov_matrix_bm, weight_bounds=(0, 0.10))
ef.max_sharpe()
weights = ef.clean_weights()
print("\nPesos ótimos (max Sharpe):")
print(pd.Series(weights).sort_values(ascending=False))

q_t    : -0.006100
omega_t: 0.000260
P_t não-zero: ABCB4    0.2
AGRO3    0.2
BBAS3    0.2
BBDC3    0.2
BRSR6    0.2
ODPV3   -0.2
POMO4   -0.2
TGMA3   -0.2
TIMS3   -0.2
UNIP6   -0.2
dtype: float64

Retornos posteriores BL:
EVEN3     0.018871
JHSF3     0.017544
EZTC3     0.015613
DIRR3     0.014632
TGMA3     0.014600
POMO4     0.014055
VLID3     0.013335
ALOS3     0.012467
VULC3     0.012418
BBDC4     0.011388
SYNE3     0.011238
UNIP6     0.011117
RANI3     0.010750
BBDC3     0.010486
PETR3     0.010232
PETR4     0.010045
KEPL3     0.009750
ITUB4     0.009574
BBAS3     0.009542
BRSR6     0.009308
CMIG4     0.009302
ITSA4     0.009100
CSMG3     0.009063
ITUB3     0.008864
LEVE3     0.008848
SAPR11    0.008613
MBRF3     0.008053
TIMS3     0.007850
ABCB4     0.007804
GRND3     0.007773
FLRY3     0.006542
ODPV3     0.006369
BRAP4     0.006336
EGIE3     0.005945
CPFE3     0.005879
TAEE11    0.005556
ISAE4     0.005527
KLBN11    0.005495
FESA4     0.005266
BBSE3     0.004922
AGRO3     0.004756